# 🎬 LMVD Dataset Processing

**LMVD**: Large-scale Multimodal Video log Database (15.2 GB, 1,823 samples)

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/DAIC-WOZ_Datasets/LMVD_Raw"
!mkdir -p "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/LMVD"
print("✅ Drive mounted")

In [ ]:
# Step 2: Install dependencies
!pip install h5py pandas tqdm scipy requests --quiet
!apt-get install -y unzip unrar > /dev/null 2>&1
print("✅ Dependencies installed")

In [ ]:
# Step 3: Download LMVD from Figshare
# Figshare file ID: 45872469
# Uses Figshare API to get actual download URL, then polls until ready

import requests
import time
from pathlib import Path
from tqdm import tqdm

FIGSHARE_FILE_ID = "45872469"
DOWNLOAD_PATH = Path("/content/lmvd_dataset")

def download_from_figshare(file_id, output_path, max_retries=10):
    """Download file from Figshare with proper handling of 202 responses."""
    
    # Try direct URL first (figstatic CDN)
    direct_url = f"https://ndownloader.figstatic.com/files/{file_id}"
    ndownloader_url = f"https://figshare.com/ndownloader/files/{file_id}"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': '*/*',
        'Accept-Language': 'en-US,en;q=0.9',
    }
    
    session = requests.Session()
    session.headers.update(headers)
    
    # Try different URLs
    urls_to_try = [direct_url, ndownloader_url]
    
    for url in urls_to_try:
        print(f"\n🔗 Trying: {url}")
        
        for attempt in range(max_retries):
            try:
                response = session.get(url, stream=True, allow_redirects=True)
                
                if response.status_code == 200:
                    total_size = int(response.headers.get('content-length', 0))
                    
                    if total_size > 1000000:  # More than 1MB
                        print(f"✅ Got download stream ({total_size / (1024**3):.2f} GB)")
                        
                        with open(output_path, 'wb') as f:
                            with tqdm(total=total_size, unit='B', unit_scale=True, desc="Downloading") as pbar:
                                for chunk in response.iter_content(chunk_size=8192*16):
                                    if chunk:
                                        f.write(chunk)
                                        pbar.update(len(chunk))
                        return True
                    else:
                        print(f"⚠️ Response too small ({total_size} bytes)")
                
                elif response.status_code == 202:
                    print(f"⏳ Server preparing file... (attempt {attempt+1}/{max_retries})")
                    time.sleep(5)  # Wait 5 seconds before retry
                    continue
                
                elif response.status_code in [301, 302, 303, 307, 308]:
                    redirect_url = response.headers.get('Location')
                    print(f"🔄 Redirect to: {redirect_url[:50]}...")
                    url = redirect_url
                    continue
                
                else:
                    print(f"❌ HTTP {response.status_code}")
                    break
                    
            except Exception as e:
                print(f"❌ Error: {e}")
                time.sleep(2)
    
    return False

print("="*60)
print("🔗 LMVD DATASET DOWNLOAD (15.2 GB)")
print("="*60)
print(f"📥 Figshare File ID: {FIGSHARE_FILE_ID}")
print("\n⬇️ Attempting download with proper Figshare handling...")

success = download_from_figshare(FIGSHARE_FILE_ID, DOWNLOAD_PATH)

if success and DOWNLOAD_PATH.exists():
    size_gb = DOWNLOAD_PATH.stat().st_size / (1024**3)
    print(f"\n✅ Download complete: {size_gb:.2f} GB")
else:
    print("\n❌ Download failed - try manual method below")

In [ ]:
# ========================================
# MANUAL FALLBACK: If automatic download fails
# ========================================
# 
# 1. Open this URL in your browser:
#    https://figshare.com/ndownloader/files/45872469
# 
# 2. When the download starts in your browser, copy the ACTUAL download URL
#    (right-click the download in Chrome → Copy Link Address)
# 
# 3. Paste it below and run this cell:

MANUAL_URL = ""  # Paste the actual download URL here

if MANUAL_URL:
    print(f"⬇️ Downloading from manual URL...")
    !wget -c -O /content/lmvd_dataset "{MANUAL_URL}"
else:
    print("💡 Paste the direct download URL above if automatic download failed")

In [ ]:
# Step 4: Extract the downloaded file
import shutil

LMVD_RAW = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/LMVD_Raw")
LMVD_OUTPUT = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/LMVD")
DOWNLOAD_PATH = Path("/content/lmvd_dataset")

if DOWNLOAD_PATH.exists() and DOWNLOAD_PATH.stat().st_size > 1000000:
    print(f"📁 File size: {DOWNLOAD_PATH.stat().st_size / (1024**3):.2f} GB")
    print("📦 Detecting format...")
    
    with open(DOWNLOAD_PATH, 'rb') as f:
        header = f.read(20)
    
    extract_dir = Path("/content/lmvd_extracted")
    extract_dir.mkdir(exist_ok=True)
    
    if header[:2] == b'PK':
        print("📦 ZIP format")
        !unzip -q -o /content/lmvd_dataset -d /content/lmvd_extracted
    elif header[:2] == b'\x1f\x8b':
        print("📦 GZIP/TAR.GZ format")
        !tar -xzf /content/lmvd_dataset -C /content/lmvd_extracted
    elif b'Rar' in header[:10]:
        print("📦 RAR format")
        !unrar x -o+ /content/lmvd_dataset /content/lmvd_extracted/
    else:
        print(f"📦 Trying common formats...")
        !tar -xf /content/lmvd_dataset -C /content/lmvd_extracted 2>/dev/null || \
         unzip -q -o /content/lmvd_dataset -d /content/lmvd_extracted 2>/dev/null
    
    files = [f for f in extract_dir.rglob("*") if f.is_file()]
    print(f"\n📂 Extracted {len(files)} files")
    
    # Show file types
    exts = {}
    for f in files:
        ext = f.suffix.lower()
        exts[ext] = exts.get(ext, 0) + 1
    for ext, count in sorted(exts.items()):
        print(f"  {ext}: {count}")
else:
    print("⚠️ Download file not found or too small")
    print("   Use the manual fallback cell above")

In [ ]:
# Step 5: Copy to Drive
extract_dir = Path("/content/lmvd_extracted")
files = [f for f in extract_dir.rglob("*") if f.is_file()]

if files:
    print(f"💾 Copying {len(files)} files to Drive...")
    file_count = 0
    for f in tqdm(files, desc="Copying"):
        dest = LMVD_RAW / f.name
        if not dest.exists():
            shutil.copy(str(f), str(dest))
            file_count += 1
    print(f"\n✅ {file_count} files saved to: {LMVD_RAW}")
else:
    print("⚠️ No files to copy")

In [ ]:
# Step 6: Process to H5
import numpy as np
import pandas as pd
from scipy.io import loadmat
import h5py
import pickle

def load_feature_file(path):
    ext = path.suffix.lower()
    if ext == '.npy': return np.load(path, allow_pickle=True)
    elif ext == '.npz':
        data = np.load(path, allow_pickle=True)
        return {k: data[k] for k in data.files}
    elif ext == '.mat': return loadmat(path)
    elif ext in ['.pkl', '.pickle']:
        with open(path, 'rb') as f: return pickle.load(f)
    elif ext in ['.h5', '.hdf5']:
        with h5py.File(path, 'r') as f:
            return {k: np.array(f[k]) for k in f.keys()}
    return None

# Load labels
labels_dict = {}
for csv_file in LMVD_RAW.rglob("*.csv"):
    try:
        df = pd.read_csv(csv_file)
        id_cols = [c for c in df.columns if any(x in c.lower() for x in ['id', 'name', 'file'])]
        label_cols = [c for c in df.columns if any(x in c.lower() for x in ['label', 'depression', 'class'])]
        if id_cols and label_cols:
            for _, row in df.iterrows():
                labels_dict[str(row[id_cols[0]])] = int(row[label_cols[0]])
            print(f"✅ Loaded {len(labels_dict)} labels from {csv_file.name}")
            break
    except: pass

# Process features
feature_files = list(LMVD_RAW.glob("*.npy")) + list(LMVD_RAW.glob("*.npz")) + \
                list(LMVD_RAW.glob("*.mat")) + list(LMVD_RAW.glob("*.pkl"))

existing_h5 = set([f.stem for f in LMVD_OUTPUT.glob("*.h5")])
to_process = [f for f in feature_files if f"lmvd_{f.stem}" not in existing_h5]

print(f"\n📂 Feature files: {len(feature_files)}")
print(f"📦 Already processed: {len(existing_h5)}")
print(f"📋 To process: {len(to_process)}")

success, fail = 0, 0
for feat_file in tqdm(to_process, desc="Processing"):
    pid = f"lmvd_{feat_file.stem}"
    h5_path = LMVD_OUTPUT / f"{pid}.h5"
    try:
        features = load_feature_file(feat_file)
        with h5py.File(h5_path, 'w') as f:
            if isinstance(features, dict):
                for k, v in features.items():
                    if isinstance(v, np.ndarray): f.create_dataset(k, data=v)
            elif isinstance(features, np.ndarray):
                f.create_dataset('features', data=features)
            if 'audio_features' not in f:
                f.create_dataset('audio_features', data=features if isinstance(features, np.ndarray) else np.zeros((1,128)))
            label = labels_dict.get(feat_file.stem, 1 if any(x in feat_file.stem.lower() for x in ['dep','patient','pos','mdd']) else 0)
            f.create_dataset('label', data=label)
            f.create_dataset('transcript', data=b'')
            f.attrs['source'] = 'LMVD'
        success += 1
    except Exception as e:
        fail += 1

print(f"\n✅ Success: {success} | ❌ Failed: {fail}")

In [ ]:
# Step 7: Create labels CSV
labels_data = []
for h5_file in LMVD_OUTPUT.glob("*.h5"):
    try:
        with h5py.File(h5_file, 'r') as f:
            label = int(f['label'][()])
        labels_data.append({
            'Participant_ID': h5_file.stem,
            'PHQ8_Score': 15 if label else 3,
            'PHQ8_Binary': label,
            'Source': 'LMVD'
        })
    except: pass

if labels_data:
    labels_df = pd.DataFrame(labels_data)
    labels_csv = "/content/drive/MyDrive/DAIC-WOZ_Datasets/lmvd_labels.csv"
    labels_df.to_csv(labels_csv, index=False)
    
    print("="*50)
    print("🏆 LMVD PROCESSING COMPLETE")
    print("="*50)
    print(f"📦 H5 Files: {len(labels_df)}")
    print(f"   Depressed: {len(labels_df[labels_df['PHQ8_Binary']==1])}")
    print(f"   Normal: {len(labels_df[labels_df['PHQ8_Binary']==0])}")
    print(f"📋 Labels: {labels_csv}")
else:
    print("⚠️ No H5 files processed yet")
    print("   Please complete the download steps first")